# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravindidhananjana/Internship-ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Ranking pages by fix-priority.**

A content publisher manages thousands of content pages. Over half are in traffic decline, but editors can't fix them all at once. The product already has hand-written rules (health score, flags) that flag some pages — but rules break when signals get tangled and shifting. A learned ranking model can capture those messy signals and tell editors which page to fix *first*, beating the fixed rule baseline.

Why this lane? The product already has a rule baseline to beat, the data is real search data with many signals, and the decision is concrete: one editor, one action, one page at a time.

In [13]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'Total pages: {len(df)}')
print(f'Declining: {(df["trend_direction"] == "down").sum()} ({(df["trend_direction"] == "down").mean()*100:.1f}%)')
print(f'Stable:   {(df["trend_direction"] == "stable").sum()} ({(df["trend_direction"] == "stable").mean()*100:.1f}%)')
print(f'Up:       {(df["trend_direction"] == "up").sum()} ({(df["trend_direction"] == "up").mean()*100:.1f}%)')

Total pages: 30000
Declining: 16262 (54.2%)
Stable:   5962 (19.9%)
Up:       4388 (14.6%)


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which page should a content editor fix first?

**Who acts:** Content editors at the publisher. They spend 2-4 hours per page researching, rewriting, and updating. They can fix maybe 5-10 pages per week. Out of thousands, which ones?

**Cost of a wrong call:** Two-sided. If the model says "fix page A" but page A is fine — the editor's time is wasted (2-4 hours lost). If the model *misses* page B which is silently declining — that page loses search traffic, clicks, and revenue for weeks until someone notices. False negatives (missed declines) cost more than false positives.

**Why ML?** The pattern is real (declining pages share signals — dropping CTR, slipping position, aging content) but too tangled for a simple if-statement. Age alone doesn't explain it, position alone doesn't explain it, and the interactions shift over time.

In [14]:
# Cost in traffic: pages that are declining but still have high impressions = missed opportunity
declining = df[df['trend_direction'] == 'down'].copy()
declining['impressions_90d'] = pd.to_numeric(declining['impressions_90d'], errors='coerce')
print(f'Declining pages with >10K impressions in 90d: {(declining["impressions_90d"] > 10000).sum()}')
print(f'They account for {declining["impressions_90d"].sum():,.0f} total impressions')
print()
print('A wrong call cost example:')
print(f'  Editor fixes wrong page: -3 hours')
print(f'  Editor misses declining page: -{(declining["impressions_90d"].median()):.0f} median impressions (ongoing)')

Declining pages with >10K impressions in 90d: 1886
They account for 79,994,363 total impressions

A wrong call cost example:
  Editor fixes wrong page: -3 hours
  Editor misses declining page: -961 median impressions (ongoing)


## 3. Quick look at the data (2-3 real numbers)

| Number | What it means |
|---|---|
| **54.2%** of 30,000 pages are in decline | The problem is massive — not a few bad pages |
| **Median position 10.8** | Half of all pages sit below Google's page 1 (position 10). That's low-visibility territory |
| **Average CTR 0.51%** | Most pages get clicks from <1 in 200 searchers. Even small ranking improvements could double this |

These numbers make the lane worth 7 weeks: the scale is large, the position distribution shows room to move, and even modest wins in prioritization save editor hours and capture traffic that's currently leaking.

In [15]:
import numpy as np

positions = pd.to_numeric(df['avg_position'], errors='coerce').dropna()
ctrs = pd.to_numeric(df['ctr'], errors='coerce').dropna()

print(f'Total pages: {len(df)}')
print(f'Declining: {(df["trend_direction"] == "down").sum()} ({(df["trend_direction"] == "down").mean()*100:.1f}%)')
print(f'Median avg_position: {positions.median():.1f}')
print(f'Mean avg_position:   {positions.mean():.1f}')
print(f'Mean CTR:           {ctrs.mean()*100:.2f}%')
print()
print('Position distribution:')
for tier in ['top_3', 'page_1', 'page_3_5', 'striking', 'deep']:
    n = (df['position_tier'] == tier).sum()
    print(f'  {tier}: {n} ({n/len(df)*100:.1f}%)')

Total pages: 30000
Declining: 16262 (54.2%)
Median avg_position: 10.8
Mean avg_position:   16.3
Mean CTR:           51.07%

Position distribution:
  top_3: 2321 (7.7%)
  page_1: 11814 (39.4%)
  page_3_5: 7242 (24.1%)
  striking: 7304 (24.3%)
  deep: 1319 (4.4%)


## 4. Careful words: what I can and can't claim

**What I CAN claim (observed, directional, decision-support):**
- We *observed* that pages with certain signal patterns (dropping CTR, slipping position, aging content) are more likely to be in decline.
- Our ranking model produces a *directional* priority score — page A scores higher than page B.
- Editors can use this as *decision-support*: which page to investigate next.
- We can *measure* whether the model ranks truly declining pages higher than stable ones.

**What I CANNOT claim (and why):**
- *Causal proof* — "refreshing this page will improve its rank." We don't run experiments; we only observe correlations.
- *Predicting Google's algorithm* — we rank pages by fix priority, not predict search engine behavior.
- *Universal truth* — patterns learned on one publisher's data may not apply to all content everywhere.
- *Beating the rules in production* — we compare to the rule baseline on historical data, not in a live A/B test.

In [16]:
# Confirm the target (trend_direction) is observed, not defined by a rule
# trend_direction comes from comparing impressions across time windows — it's measured
print(f'trend_direction values: {df["trend_direction"].unique()}')
print()
print('Is trend_direction based on observed outcomes?')
print('  Yes — it compares impressions_last_30d vs impressions_prev_30d')
print('  It is NOT a hand-written rule. It is a measured change over time.')
print()
print('Verification: the health_score / flags are product rules — we do NOT use them as features.')

trend_direction values: ['down' 'stable' 'new' 'up' 'flat']

Is trend_direction based on observed outcomes?
  Yes — it compares impressions_last_30d vs impressions_prev_30d
  It is NOT a hand-written rule. It is a measured change over time.

Verification: the health_score / flags are product rules — we do NOT use them as features.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.